# M5 IA Agêntica - Equipe de Pesquisa de Mercado

## 1. Introdução

### 1.1. Visão Geral do Laboratório

Neste laboratório, você assumirá o papel de um **líder técnico de IA em uma marca de moda** preparando uma campanha de óculos de sol para o verão. Sua tarefa é projetar um **pipeline criativo totalmente automatizado** que espelhe um cenário de negócios do mundo real. Em vez de lidar com cada peça manualmente, você guiará um sistema que varre fontes online em busca de tendências de moda emergentes, combina essas tendências com óculos do catálogo interno, projeta um visual de campanha, gera uma frase de marketing curta e, finalmente, empacota tudo em um **relatório pronto para executivos**.  

O objetivo é experimentar como múltiplos agentes, ferramentas e modelos podem ser orquestrados em um fluxo de trabalho único e coerente. Ao final deste laboratório, você terá construído um pipeline que parece menos um script de passos isolados e mais uma pequena equipe trabalhando junta para resolver um desafio criativo.  

### 1.2. 🎯 Resultado de Aprendizado

Ao concluir este laboratório, você verá como ir além de interações de turno único com um modelo e, em vez disso, projetar **pipelines multi-agente** que coordenam planejamento, pesquisa e geração criativa. Você aprenderá como basear o raciocínio do agente em ferramentas externas para que as saídas não sejam apenas imaginativas, mas também apoiadas por dados reais. Você também experimentará etapas de reflexão e empacotamento que impõem controle de qualidade e preparam resultados para um público executivo.  

Em resumo, este laboratório é sobre aprender como combinar a **imaginação de grandes modelos de linguagem** com a **disciplina de fluxos de trabalho estruturados**, dando a você um padrão prático para construir sistemas autônomos que sejam criativos e confiáveis.

## 2. Configuração: Importar bibliotecas e carregar ambiente

Como nos laboratórios anteriores, agora você importa as bibliotecas necessárias, carrega as variáveis de ambiente e configura os utilitários auxiliares.

In [ ]:
# =========================
# Imports
# =========================

# --- Standard library ---
import base64
import json
import os
import re
import sys
from datetime import datetime
from io import BytesIO

# --- Third-party ---
import requests
import openai
from PIL import Image
from dotenv import load_dotenv
from IPython.display import Markdown, display
import aisuite

# --- Local / project ---
sys.path.append(os.path.abspath('../../'))
from utils import tools
from utils import utils


# =========================
# Environment & Client
# =========================
load_dotenv()
client = aisuite.Client()


## 3. Ferramentas Disponíveis

Pipelines agênticos só se tornam eficazes quando o modelo recebe **capacidades explícitas** além de seu raciocínio base. Declarar essas ferramentas antecipadamente torna o espaço de ação do agente inequívoco, garante que os prompts guiem naturalmente a seleção de ferramentas e mantém a orquestração e os testes transparentes por meio de interfaces bem definidas.  

Você montará uma **equipe de pesquisa de marketing**, um grupo de agentes especializados colaborando para projetar uma campanha de óculos de sol para o verão. Para capacitá-los, começamos definindo as ferramentas que basearão seu raciocínio em dados reais.  

A primeira ferramenta é `tools.tavily_search_tool`, que realiza pesquisas na web ao vivo para descobrir evidências das tendências atuais da moda. Experimente agora rodando uma consulta simples para *“trends in sunglasses fashion”*:

In [ ]:
tools.tavily_search_tool('trends in sunglasses fashion')

A segunda ferramenta é `tools.product_catalog_tool`, que retorna o catálogo interno de óculos de sol. Cada entrada inclui detalhes como nome do produto, ID, descrição, quantidade em estoque e preço. Esses dados estruturados permitirão que os agentes conectem as tendências de moda online com itens reais em estoque:

In [ ]:
tools.product_catalog_tool()

Com essas ferramentas no lugar, você definiu um espaço de ação claro e fontes de dados confiáveis. Na próxima seção, você construirá os agentes que as usam para transformar sinais brutos de moda em insights estruturados e ativos de campanha.

## 4. Definições de Agentes — Construindo Sua Equipe

Agora que você definiu as ferramentas, é hora de colocá-las para trabalhar. Nesta fase, você montará uma **equipe de pesquisa de marketing**, um grupo de agentes especializados que você dirige com instruções naturais.  

Cada agente conta com as ferramentas que você introduziu anteriormente, e juntos eles transformam dados brutos de tendências em um relatório de campanha polido. Nós os definiremos um por um, apresentando seu papel e mostrando o código que implementa cada um.

### 4.1. Agente de Pesquisa de Mercado (Market Research Agent)

Com o **Agente de Pesquisa de Mercado**, você dá o primeiro passo na construção de sua campanha. Você pede que ele varra a web com `tavily_search_tool` e descubra o que está em alta na moda de óculos de sol agora. Em seguida, você o orienta a verificar esses sinais em relação ao seu catálogo interno usando `product_catalog_tool`, para que você saiba quais dos seus produtos se encaixam no momento.  

O agente devolve um resumo conciso: os principais insights de moda que encontrou, os produtos que se alinham com eles e uma breve explicação de por que essas escolhas fazem sentido para o seu empurrão de verão. Isso lhe dá uma base clara e orientada por dados para moldar o restante da campanha.  

Você pode agora rodar a seguinte célula para definir o **Agente de Pesquisa de Mercado** em código.

In [ ]:
def market_research_agent(return_messages: bool = False):

    utils.log_agent_title_html("Market Research Agent", "🕵️‍♂️")

    prompt_ = f"""
You are a fashion market research agent tasked with preparing a trend analysis for a summer sunglasses campaign.

Your goal:
1. Explore current fashion trends related to sunglasses using web search.
2. Review the internal product catalog to identify items that align with those trends.
3. Recommend one or more products from the catalog that best match emerging trends.
4. If needed, today date is {datetime.now().strftime("%Y-%m-%d")}.

You can call the following tools:
- tavily_search_tool: to discover external web trends.
- product_catalog_tool: to inspect the internal sunglasses catalog.

Once your analysis is complete, summarize:
- The top 2–3 trends you found.
- The product(s) from the catalog that fit these trends.
- A justification of why they are a good fit for the summer campaign.
"""
    messages = [{"role": "user", "content": prompt_}]
    tools_ = tools.get_available_tools()

    while True:
        response = client.chat.completions.create(
            model="openai:gpt-4o-mini",
            messages=messages,
            tools=tools_,
            tool_choice="auto"
        )

        msg = response.choices[0].message

        if msg.content:
            utils.log_final_summary_html(msg.content)
            return (msg.content, messages) if return_messages else msg.content

        if msg.tool_calls:
            for tool_call in msg.tool_calls:
                utils.log_tool_call_html(tool_call.function.name, tool_call.function.arguments)
                result = tools.handle_tool_call(tool_call)
                utils.log_tool_result_html(result)

                messages.append(msg)
                messages.append(tools.create_tool_response_message(tool_call, result))
        else:
            utils.log_unexpected_html()
            return ("[⚠️ Unexpected: No tool_calls or content returned]", messages) if return_messages else "[⚠️ Unexpected: No tool_calls or content returned]"

Vamos tentar obter alguns conselhos do **Agente de Pesquisa de Mercado** sobre nossa campanha de óculos de sol para o verão.

In [ ]:
market_research_result = market_research_agent()

Em seguida, você transformará esse resumo em um conceito visual com o Agente Designer Gráfico.

### 4.2. Agente Designer Gráfico (Graphic Designer Agent)

Com o **Agente Designer Gráfico**, você passa da análise para a criatividade.  
Você pega o resumo do seu Agente de Pesquisa de Mercado e pede a este para traduzi-lo em um conceito visual.  
Como o `aisuite` ainda não suporta geração direta de imagem (como DALL·E), você guia o processo em dois estágios:  

1. Primeiro, o agente usa `aisuite` com um modelo de texto da OpenAI (`gpt-4o-mini`) para criar um **prompt** vívido e uma **legenda** curta e envolvente.  
2. Então, o prompt é enviado para a API `dall-e-3` da OpenAI para gerar a **imagem da campanha** em si.  

O resultado lhe dá tudo o que você precisa para seguir em frente: a imagem gerada (salva localmente para reutilização), o prompt exato que a produziu (útil para iteração) e uma legenda polida para a narrativa da campanha.  

<div style="border:1px solid #fca5a5; border-left:6px solid #ef4444; background:#fee2e2; border-radius:6px; padding:12px 14px; color:#111827; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">
  <strong>Nota:</strong> Neste ponto, o <code>aisuite</code> <strong>não suporta geração direta de imagem</strong>.  
  É por isso que você combina sua saída baseada em texto (prompt + legenda) com o <code>dall-e-3</code> da OpenAI para produzir o visual final da campanha.
</div>  

Você pode agora rodar a seguinte célula para definir o **Agente Designer Gráfico** em código.

In [ ]:
def graphic_designer_agent(trend_insights: str, caption_style: str = "short punchy", size: str = "1024x1024") -> dict:

    """
    Uses aisuite to generate a marketing prompt/caption and OpenAI (directly) to generate the image.

    Args:
        trend_insights (str): Trend summary from the researcher agent.
        caption_style (str): Optional style hint for caption.
        size (str): Image resolution (e.g., '1024x1024').

    Returns:
        dict: A dictionary with image_url, prompt, and caption.
    """

    utils.log_agent_title_html("Graphic Designer Agent", "🎨")

    # Step 1: Generate prompt and caption using aisuite
    system_message = (
        "You are a visual marketing assistant. Based on the input trend insights, "
        "write a creative and visual prompt for an AI image generation model, and also a short caption."
    )

    user_prompt = f"""
Trend insights:
{trend_insights}

Please output:
1. A vivid, descriptive prompt to guide image generation.
2. A marketing caption in style: {caption_style}.

Respond in this format:
{{"prompt": "...", "caption": "..."}}
"""

    chat_response = client.chat.completions.create(
        model="openai:gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_prompt}
        ]
    )

    content = chat_response.choices[0].message.content.strip()
    match = re.search(r'\{.*\}', content, re.DOTALL)
    parsed = json.loads(match.group(0)) if match else {"error": "No JSON returned", "raw": content}

    prompt = parsed["prompt"]
    caption = parsed["caption"]

    # Step 2: Generate image directly using openai-python
    openai_client = openai.OpenAI()

    image_response = openai_client.images.generate(
        model="dall-e-3",
        prompt=prompt,
        size=size,
        quality="standard",
        n=1,
        response_format="url"
    )

    image_url = image_response.data[0].url

    # Save image locally
    img_bytes = requests.get(image_url).content
    img = Image.open(BytesIO(img_bytes))

    filename = os.path.basename(image_url.split("?")[0])
    image_path = filename
    img.save(image_path)


    # Log summary with local image
    utils.log_final_summary_html(f"""
        <h3>Generated Image and Caption</h3>

        <p><strong>Image Path:</strong> <code>{image_path}</code></p>

        <p><strong>Generated Image:</strong></p>
        <img src="{image_path}" alt="Generated Image" style="max-width: 100%; height: auto; border: 1px solid #ccc; border-radius: 8px; margin-top: 10px; margin-bottom: 10px;">

        <p><strong>Prompt:</strong> {prompt}</p>
    """)


    return {
        "image_url": image_url,
        "prompt": prompt,
        "caption": caption,
        "image_path": image_path  
    }



Agora vamos rodar o `graphic_designer_agent` para gerar uma imagem de campanha, usando os insights de tendências fornecidos pelo **Agente de Pesquisa de Mercado**.

In [ ]:
graphic_designer_agent_result = graphic_designer_agent(
    trend_insights=market_research_result,
)


Com um visual em mãos, você criará a voz da campanha usando o Agente Redator.

### 4.3. Agente Redator (Copywriter Agent)

Uma vez que o **Agente de Pesquisa de Mercado** e o **Agente Designer Gráfico** tenham feito seu trabalho, agora você se volta para o **Agente Redator**. Com a imagem da campanha e o resumo das tendências em mãos, você pede a este agente para criar a voz de sua campanha.  

Ele aceita o visual e a análise juntos como entrada multimodal e cria uma frase de marketing curta e elegante que captura a essência da mensagem. Junto com a frase, ele lhe dá uma justificativa clara — por que a frase se encaixa na imagem e como ela se conecta com as tendências.  

Dessa forma, você não recebe apenas uma frase de efeito, você também recebe o raciocínio por trás dela, tornando mais fácil defender e refinar diante das partes interessadas (stakeholders).

In [ ]:
def copywriter_agent(image_path: str, trend_summary: str, model: str = "openai:gpt-4o-mini") -> dict:

    """
    Uses aisuite (OpenAI only) to send an image and a trend summary and return a campaign quote.

    Args:
        image_path (str): URL of the image to be analyzed.
        trend_summary (str): Text from the researcher agent.
        model (str): OpenAI model (e.g., openai:gpt-4o-mini, openai:gpt-4o)

    Returns:
        dict: {
            "quote": "...",
            "justification": "...",
            "image_path": "..."
        }
    """

    utils.log_agent_title_html("Copywriter Agent", "✍️")

    # Step 1: Load local image and encode as base64
    with open(image_path, "rb") as f:
        img_bytes = f.read()

    b64_img = base64.b64encode(img_bytes).decode("utf-8")

    # Step 2: Build OpenAI-compliant multimodal message
    messages = [
        {
            "role": "system",
            "content": "You are a copywriter that creates elegant campaign quotes based on an image and a marketing trend summary."
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/png;base64,{b64_img}",
                        "detail": "auto"
                    }
                },
                {
                    "type": "text",
                    "text": f"""
Here is a visual marketing image and a trend analysis:

Trend summary:
\"\"\"{trend_summary}\"\"\"

Please return a JSON object like:
{{
  "quote": "A short, elegant campaign phrase (max 12 words)",
  "justification": "Why this quote matches the image and trend"
}}"""
                }
            ]
        }
    ]

    # Step 3: Send request via aisuite
    response = client.chat.completions.create(
        model=model,
        messages=messages,
    )

    # Step 4: Parse JSON response
    content = response.choices[0].message.content.strip()

    utils.log_final_summary_html(content)

    try:
        match = re.search(r'\{.*\}', content, re.DOTALL)
        parsed = json.loads(match.group(0)) if match else {"error": "No valid JSON returned"}
    except Exception as e:
        parsed = {"error": f"Failed to parse: {e}", "raw": content}


    parsed["image_path"] = image_path
    return parsed


Em seguida, vamos chamar o Agente Redator para gerar uma citação curta de campanha com base na imagem de marketing e nos insights de tendências produzidos anteriormente.

In [ ]:
copywriter_agent_result = copywriter_agent(
    image_path=graphic_designer_agent_result["image_path"],
    trend_summary=market_research_result,
)

Com uma frase e justificativa prontas, você empacotará tudo em um artefato pronto para executivos usando o Agente de Empacotamento.

### 4.4. Agente de Empacotamento (Packaging Agent)

Finalmente, você traz o **Agente de Empacotamento** para amarrar tudo. Depois que o **Agente de Pesquisa de Mercado**, **Agente Designer Gráfico** e **Agente Redator** contribuíram com sua parte, este agente compila toda a história em um artefato polido.  

Você pede a ele que pegue o resumo das tendências, o visual da campanha, a frase criada e a justificativa, e os reúna em um relatório markdown pronto para executivos. Ao longo do caminho, ele reescreve os insights de tendências para clareza e tom, garante que a frase seja estilizada corretamente com a imagem e organiza tudo para que o documento final pareça profissional e persuasivo.  

Com essa etapa, você termina com um pacote de campanha completo — fácil de compartilhar, visualmente envolvente e pronto para revisão de nível CEO.

In [ ]:
def packaging_agent(trend_summary: str, image_url: str, quote: str, justification: str, output_path: str = "campaign_summary.md") -> str:

    """
    Packages the campaign assets into a beautifully formatted markdown report for executive review.

    Args:
        trend_summary (str): Summary of the market trends.
        image_url (str): URL of the campaign image.
        quote (str): Marketing quote to overlay.
        justification (str): Explanation for the quote.
        output_path (str): Path to save the markdown report.

    Returns:
        str: Path to the saved markdown file.
    """

    utils.log_agent_title_html("Packaging Agent", "📦")

    # We use this path in the src of the <img>
    styled_image_html = f"""
![Open the generated file to see]({image_url})
    """

    beautified_summary = client.chat.completions.create(
        model="openai:gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a marketing communication expert writing elegant campaign summaries for executives."},
            {"role": "user", "content": f"""
Please rewrite the following trend summary to be clear, professional, and engaging for a CEO audience:

\"\"\"{trend_summary.strip()}\"\"\"
"""}
        ]
    ).choices[0].message.content.strip()

    utils.log_tool_result_html(beautified_summary)

    # Combine all parts into markdown
    markdown_content = f"""# 🕶️ Summer Sunglasses Campaign – Executive Summary

## 📊 Refined Trend Insights
{beautified_summary}

## 🎯 Campaign Visual
{styled_image_html}

## ✍️ Campaign Quote
{quote.strip()}

## ✅ Why This Works
{justification.strip()}

---

*Report generated on {datetime.now().strftime('%Y-%m-%d')}*
"""

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(markdown_content)

    return output_path



Com seu resumo de tendências, imagem de campanha e frase prontos, agora você entrega tudo ao **Agente de Empacotamento**. O trabalho dele é reunir essas peças em um relatório polido e pronto para executivos. Rode a próxima célula para gerá-lo.

In [ ]:
packaging_agent_result = packaging_agent(
    trend_summary=market_research_result,
    image_url=graphic_designer_agent_result["image_path"],
    quote=copywriter_agent_result["quote"],
    justification=copywriter_agent_result["justification"],
    output_path=f"campaign_summary_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.md"
)

O resultado final será um relatório de campanha lindamente formatado que você pode visualizar diretamente no notebook. Ele incluirá:  

- Um resumo de tendências refinado que você verá reescrito para clareza executiva  
- Uma imagem visualmente estilizada com sua frase de campanha sobreposta usando HTML  
- Uma justificativa clara para você entender por que o visual e a mensagem se alinham com as tendências atuais  
- Um carimbo de data/hora mostrando exatamente quando o relatório foi gerado  

Você pode vê-lo com:

In [ ]:
# Load and render the markdown content
with open(packaging_agent_result, "r", encoding="utf-8") as f:
    md_content = f.read()

display(Markdown(md_content))


Finalmente, você embrulhará todo o fluxo de trabalho em uma única função chamável para executar todo o pipeline em um passo.

## 5. Pipeline de Campanha Completo – `run_sunglasses_campaign_pipeline`

Nesta etapa, você definirá uma única função, `run_sunglasses_campaign_pipeline`, que une todas as peças em um fluxo de trabalho perfeito para sua campanha de óculos de sol de verão.  

A função irá:  
- Executar pesquisa de mercado para varrer tendências de moda e combiná-las com seu catálogo.  
- Gerar uma imagem e legenda visualmente estilizadas.  
- Criar uma frase de campanha curta e elegante com justificativa.  
- Empacotar tudo em um relatório markdown polido feito para revisão executiva.  

Ao definir essa função, você torna fácil rodar o **pipeline inteiro em uma chamada** enquanto ainda é capaz de rastrear resultados intermediários e ver o relatório final.

In [ ]:
def run_sunglasses_campaign_pipeline(output_path: str = "campaign_summary.md") -> dict:
    """
    Runs the full summer sunglasses campaign pipeline:
    1. Market research (search trends + match products)
    2. Generate visual + caption
    3. Generate quote based on image + trend
    4. Create executive markdown report

    Returns:
        dict: Dictionary containing all intermediate results + path to final report
    """
    # 1. Run market research agent
    trend_summary = market_research_agent()
    print("✅ Market research completed")

    # 2. Generate image + caption
    visual_result = graphic_designer_agent(trend_insights=trend_summary)
    image_path = visual_result["image_path"]
    print("🖼️ Image generated")

    # 3. Generate quote based on image + trends
    quote_result = copywriter_agent(image_path=image_path, trend_summary=trend_summary)
    quote = quote_result.get("quote", "")
    justification = quote_result.get("justification", "")
    print("💬 Quote created")

    # 4. Generate markdown report
    md_path = packaging_agent(
        trend_summary=trend_summary,
        image_url=image_path,  
        quote=quote,
        justification=justification,
        output_path=f"campaign_summary_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.md"
    )

    print(f"📦 Report generated: {md_path}")

    return {
        "trend_summary": trend_summary,
        "visual": visual_result,
        "quote": quote_result,
        "markdown_path": md_path
    }


Você pode agora criar um relatório de campanha completo rodando o pipeline em uma única chamada. Apenas execute a próxima célula:

In [ ]:
results = run_sunglasses_campaign_pipeline()

### 5.1. Resultados

Execute a célula seguinte para ver as saídas geradas pelo pipeline completo da campanha.

In [ ]:
# Load and render the markdown content
with open(results["markdown_path"], "r", encoding="utf-8") as f:
    md_content = f.read()

display(Markdown(md_content))

Abaixo você também pode inspecionar todos os resultados intermediários:

In [ ]:
results

## 6. Conclusão

Neste laboratório, você construiu um **pipeline multi-agente** que automatiza um fluxo de trabalho criativo complexo. Em vez de escrever um único script rígido, você orquestrou uma equipe de agentes especializados:

- O **Agente de Pesquisa de Mercado** ancorou as decisões em dados do mundo real.
- O **Agente Designer Gráfico** transformou insights em conceitos visuais.
- O **Agente Redator** criou mensagens que conectam as duas coisas.
- O **Agente de Empacotamento** entregou um produto final polido.

Essa abordagem mostra como a IA Agêntica pode ir além de tarefas simples de chat para lidar com todo o ciclo de vida de um processo de análise, criação e relatórios de negócios.